# Run Analysis\n\nReproducible proof notebook for three core claims: threshold estimation, patching-rate sensitivity, and strategy comparison.

In [ ]:
from __future__ import annotations\n
\n
import numpy as np\n
import pandas as pd\n
import seaborn as sns\n
import matplotlib.pyplot as plt\n
from scipy.optimize import curve_fit\n
\n
from models.university_network import UniversityNetwork\n
\n
sns.set_theme(style='whitegrid')\n
%matplotlib inline\n
\n
def logistic(x: np.ndarray, L: float, x0: float, k: float) -> np.ndarray:\n
    return L / (1 + np.exp(-k * (x - x0)))\n
\n
def run_model(\n
    virus_spread_chance: float,\n
    patching_rate: float,\n
    patching_strategy: str,\n
    steps: int,\n
    seed: int,\n
    num_agents: int = 200,\n
    initial_outbreak_size: int = 5,\n
    attachment_parameter: int = 3,\n
) -> pd.DataFrame:\n
    model = UniversityNetwork(\n
        num_agents=num_agents,\n
        initial_outbreak_size=initial_outbreak_size,\n
        virus_spread_chance=virus_spread_chance,\n
        patching_rate=patching_rate,\n
        patching_strategy=patching_strategy,\n
        attachment_parameter=attachment_parameter,\n
        seed=seed,\n
    )\n
    for _ in range(steps):\n
        model.step()\n
    return model.datacollector.get_model_vars_dataframe()\n

## Experiment 1: Percolation threshold sweep\n
\n
Sweep transmission from `0.02` to `1.00` (50 values), run 50 trials each with no intervention (`patching_rate=0.0`), and fit a logistic curve to outbreak probability where outbreak is `CumulativeExposed > 100`.

In [ ]:
spread_values = np.round(np.arange(0.02, 1.001, 0.02), 2)\ntrials = 50\noutbreak_probs = []\n\nfor spread in spread_values:\n    outbreak_count = 0\n    for trial in range(trials):\n        history = run_model(\n            virus_spread_chance=float(spread),\n            patching_rate=0.0,\n            patching_strategy='Random',\n            steps=150,\n            seed=trial,\n            num_agents=200,\n            initial_outbreak_size=5,\n            attachment_parameter=3,\n        )\n        ever_exposed = int(history['CumulativeExposed'].iloc[-1])\n        if ever_exposed > 100:\n            outbreak_count += 1\n    outbreak_probs.append(outbreak_count / trials)\n\nx = np.array(spread_values, dtype=float)\ny = np.array(outbreak_probs, dtype=float)\n\nparams, _ = curve_fit(\n    logistic,\n    x,\n    y,\n    p0=[1.0, 0.30, 10.0],\n    bounds=([0.5, 0.0, 0.01], [1.5, 1.0, 200.0]),\n    maxfev=50000,\n)\nL_hat, critical_spread, k_hat = params\n\nx_fit = np.linspace(x.min(), x.max(), 400)\ny_fit = logistic(x_fit, *params)\n\nplt.figure(figsize=(10, 5))\nsns.scatterplot(x=x, y=y, s=40, color='#1f77b4', label='Monte Carlo outbreak probability')\nplt.plot(x_fit, y_fit, color='#d62728', linewidth=2, label='Logistic fit')\nplt.axvline(critical_spread, color='black', linestyle='--', linewidth=1.5, label=f'Critical ≈ {critical_spread:.3f}')\nplt.text(critical_spread + 0.01, 0.52, f'critical={critical_spread:.3f}', fontsize=10)\nplt.ylim(-0.02, 1.02)\nplt.xlabel('virus_spread_chance')\nplt.ylabel('P(outbreak: CumulativeExposed > 100)')\nplt.title('Percolation Threshold on BA(200, m=3) Network')\nplt.legend()\nplt.tight_layout()\nplt.show()\n\nprint(f'Critical transmission rate (logistic midpoint): {critical_spread:.3f}')\nprint(f'Fitted logistic parameters: L={L_hat:.3f}, k={k_hat:.3f}')\n

## Experiment 2: Patching rate comparison\n
\n
This reproduces the ~26.7% reduction from doubling the patching rate under random selection.

In [ ]:
def collect_peak_infected(\n
    patching_rate: float,\n
    patching_strategy: str,\n
    virus_spread_chance: float = 0.4,\n
    runs: int = 50,\n
    steps: int = 250,\n
) -> list[float]:\n
    peaks = []\n
    for trial in range(runs):\n
        history = run_model(\n
            virus_spread_chance=virus_spread_chance,\n
            patching_rate=patching_rate,\n
            patching_strategy=patching_strategy,\n
            steps=steps,\n
            seed=trial,\n
            num_agents=200,\n
            initial_outbreak_size=5,\n
            attachment_parameter=3,\n
        )\n
        peaks.append(float(history['Infected'].max()))\n
    return peaks\n
\n
peaks_rate_005 = collect_peak_infected(patching_rate=0.05, patching_strategy='Random')\n
peaks_rate_010 = collect_peak_infected(patching_rate=0.10, patching_strategy='Random')\n
\n
mean_005 = float(np.mean(peaks_rate_005))\n
mean_010 = float(np.mean(peaks_rate_010))\n
rate_reduction_pct = ((mean_005 - mean_010) / mean_005) * 100\n
\n
print(f'Mean peak (patching_rate=0.05): {mean_005:.2f}')\n
print(f'Mean peak (patching_rate=0.10): {mean_010:.2f}')\n
print(f'Peak infection reduction from 0.05 -> 0.10: {rate_reduction_pct:.2f}%')\n
\n
rate_df = pd.DataFrame({\n
    'PeakInfected': peaks_rate_005 + peaks_rate_010,\n
    'PatchingRate': ['0.05'] * len(peaks_rate_005) + ['0.10'] * len(peaks_rate_010),\n
})\n
\n
plt.figure(figsize=(8, 5))\n
sns.boxplot(data=rate_df, x='PatchingRate', y='PeakInfected', palette=['#1f77b4', '#ff7f0e'])\n
sns.stripplot(data=rate_df, x='PatchingRate', y='PeakInfected', color='black', alpha=0.35, size=3)\n
plt.title('Effect of Patching Rate on Peak Infection (Random Strategy)')\n
plt.xlabel('Patching rate')\n
plt.ylabel('Peak infected count')\n
plt.tight_layout()\n
plt.show()\n

## Experiment 3: Strategy comparison\n
\n
This isolates the effect of patching strategy while holding rate constant.

In [ ]:
peaks_random = collect_peak_infected(\n
    patching_rate=0.10,\n
    patching_strategy='Random',\n
    virus_spread_chance=0.4,\n
    runs=50,\n
    steps=250,\n
)\n
peaks_targeted = collect_peak_infected(\n
    patching_rate=0.10,\n
    patching_strategy='Targeted',\n
    virus_spread_chance=0.4,\n
    runs=50,\n
    steps=250,\n
)\n
\n
mean_random = float(np.mean(peaks_random))\n
mean_targeted = float(np.mean(peaks_targeted))\n
strategy_reduction_pct = ((mean_random - mean_targeted) / mean_random) * 100\n
\n
print(f'Mean peak (Random): {mean_random:.2f}')\n
print(f'Mean peak (Targeted): {mean_targeted:.2f}')\n
print(f'Peak infection reduction Random -> Targeted: {strategy_reduction_pct:.2f}%')\n
\n
strategy_df = pd.DataFrame({\n
    'PeakInfected': peaks_random + peaks_targeted,\n
    'Strategy': ['Random'] * len(peaks_random) + ['Targeted'] * len(peaks_targeted),\n
})\n
\n
plt.figure(figsize=(8, 5))\n
sns.boxplot(data=strategy_df, x='Strategy', y='PeakInfected', palette=['#6baed6', '#31a354'])\n
sns.stripplot(data=strategy_df, x='Strategy', y='PeakInfected', color='black', alpha=0.35, size=3)\n
plt.title('Random vs Targeted Hub Patching at Rate 0.10')\n
plt.xlabel('Patching strategy')\n
plt.ylabel('Peak infected count')\n
plt.tight_layout()\n
plt.show()\n